# Data collection for the year 2013

In [1]:
import cocopp
dsl = cocopp.load("bbob/2013/*")

In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> HCMA_loshchilov  (ERT=6 @ 1e-08)
dim= 2, F 2 -> HMLSL_pal  (ERT=84.8 @ 1e-08)
dim= 2, F 3 -> BIPOP-aCMA-STEP_loshchilov  (ERT=510 @ 1e-08)
dim= 2, F 4 -> HMLSL_pal  (ERT=2.68e+03 @ 1e-08)
dim= 2, F 5 -> SMAC-BBOB_hutter  (ERT=4.93 @ 1e-08)
dim= 2, F 6 -> simplex_pal  (ERT=224 @ 1e-08)
dim= 2, F 7 -> lmm-CMA-ES_auger  (ERT=177 @ 1e-08)
dim= 2, F 8 -> fminunc_pal  (ERT=94.4 @ 1e-08)
dim= 2, F 9 -> HMLSL_pal  (ERT=58.3 @ 1e-08)
dim= 2, F10 -> lmm-CMA-ES_auger  (ERT=159 @ 1e-08)
dim= 2, F11 -> lmm-CMA-ES_auger  (ERT=160 @ 1e-08)
dim= 2, F12 -> simplex_pal  (ERT=292 @ 1e-08)
dim= 2, F13 -> simplex_pal  (ERT=251 @ 1e-08)
dim= 2, F14 -> simplex_pal  (ERT=175 @ 1e-08)
dim= 2, F15 -> lmm-CMA-ES_auger  (ERT=1.01e+03 @ 1e-08)
dim= 2, F16 -> lmm-CMA-ES_auger  (ERT=621 @ 1e-08)
dim= 2, F17 -> IP-10DDr_liao  (ERT=1.65e+03 @ 1e-08)
dim= 2, F18 -> lmm-CMA-ES_auger  (ERT=2.8e+03 @ 1e-08)
dim= 2, F19 -> HMLSL_pal  (ERT=84.9 @ 1e-08)
dim= 2, F20 -> HMLSL_pal  (ERT=754 @ 1e-08)
dim= 2, F21 

In [3]:
from collections import Counter, defaultdict

In [4]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

BIPOP-saACM-k_loshchilov: 28
BIPOP-aCMA-STEP_loshchilov: 18
lmm-CMA-ES_auger: 17
HCMA_loshchilov: 12
texp_liao: 11
HMLSL_pal: 8
fminunc_pal: 8
simplex_pal: 7
SMAC-BBOB_hutter: 5
MEMPSODE_voglis: 5
MLSL_pal: 5
tany_liao: 5
fmincon_pal: 4
IP-10DDr_liao: 3
OQNLP_pal: 3
IPOP400D_auger: 2
IP_liao: 2
CMAES_Hutter_hutter: 1


In [5]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'lmm-CMA-ES_auger',
 3: 'lmm-CMA-ES_auger',
 5: 'BIPOP-aCMA-STEP_loshchilov',
 10: 'BIPOP-saACM-k_loshchilov',
 20: 'BIPOP-saACM-k_loshchilov',
 40: 'BIPOP-saACM-k_loshchilov'}

In [6]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target              best_algorithm  \
0           2            1  1.000000e-08             HCMA_loshchilov   
1           2            1  1.000000e-05             HCMA_loshchilov   
2           2            1  1.000000e-03             HCMA_loshchilov   
3           2            1  1.000000e-02             HCMA_loshchilov   
4           2            1  1.000000e-01             HCMA_loshchilov   
5           2            2  1.000000e-08                   HMLSL_pal   
6           2            2  1.000000e-05                   HMLSL_pal   
7           2            2  1.000000e-03                   HMLSL_pal   
8           2            2  1.000000e-02                   HMLSL_pal   
9           2            2  1.000000e-01                   HMLSL_pal   
10          2            3  1.000000e-08  BIPOP-aCMA-STEP_loshchilov   
11          2            3  1.000000e-05  BIPOP-aCMA-STEP_loshchilov   
12          2 

In [8]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2013.csv", index=False)
